In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from datetime import date, timedelta
import math
import polars as pl

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

VAL_START = date(2020, 9, 9)
REFERENCE_DATE = VAL_START - timedelta(days=1)

K = 12
HISTORY_DAYS = 56

In [3]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

actuals = validation_ground_truth["actual"].to_list()
catalog_size = article_mapping.height

print("Train:", train.select(pl.len()).collect().item())
print("Validation users:", validation_ground_truth.height)
print("Catalog size:", catalog_size)

Train: 31292772
Validation users: 72019
Catalog size: 105542


## Decay Popularity

In [4]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i

        seen.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(1 / math.log2(i + 2) for i, item in enumerate(predicted[:k]) if item in actual)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))

    return dcg / idcg


def evaluate_static(actuals, predicted, catalog_size, k=12):
    return {
        "MAP@12": sum(average_precision_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "Recall@12": sum(recall_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "NDCG@12": sum(ndcg_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "Coverage": len(set(predicted[:k])) / catalog_size
    }

In [5]:
decay_source = train.with_columns(
    (pl.lit(REFERENCE_DATE) - pl.col("t_dat"))
    .dt.total_days()
    .cast(pl.Float32)
    .alias("age_days")
)

In [6]:
decay_results = []

for half_life in [3, 7, 14, 28, 56]:
    top12 = (
        decay_source
        .with_columns(
            (-(pl.col("age_days") * math.log(2) / half_life))
            .exp()
            .alias("weight")
        )
        .group_by("article_idx")
        .agg(pl.col("weight").sum().alias("score"))
        .sort("score", descending=True)
        .head(K)
        .collect()["article_idx"]
        .to_list()
    )

    decay_results.append({
        "half_life": half_life,
        **evaluate_static(actuals, top12, catalog_size)
    })

In [7]:
decay_results = pl.DataFrame(decay_results).sort("MAP@12", descending=True)

decay_results

half_life,MAP@12,Recall@12,NDCG@12,Coverage
i64,f64,f64,f64,f64
3,0.007,0.022814,0.013504,0.000114
7,0.006617,0.022939,0.013277,0.000114
14,0.006221,0.022112,0.012763,0.000114
28,0.004956,0.018078,0.0105,0.000114
56,0.003849,0.011072,0.007503,0.000114


In [8]:
BEST_HALF_LIFE = decay_results["half_life"][0]

print("Best half-life:", BEST_HALF_LIFE)

Best half-life: 3


## Personal History + Decay Popularity

In [9]:
decay_top100 = (
    decay_source
    .with_columns(
        (-(pl.col("age_days") * math.log(2) / BEST_HALF_LIFE))
        .exp()
        .alias("weight")
    )
    .group_by("article_idx")
    .agg(pl.col("weight").sum().alias("score"))
    .sort("score", descending=True)
    .head(100)
    .collect()["article_idx"]
    .to_list()
)

decay_top100[:12]

[103794,
 67523,
 67544,
 53893,
 104046,
 103797,
 3092,
 94675,
 101368,
 101719,
 103187,
 71111]

In [10]:
val_users = validation_ground_truth.select("customer_idx")

user_history = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=HISTORY_DAYS))
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"], descending=[False, True])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").unique(maintain_order=True).head(K).alias("history"))
    .collect()
)

user_history.head()

customer_idx,history
u32,list[u32]
39,"[100028, 105274, … 17132]"
87,"[103584, 102473, … 76209]"
180,[81825]
204,"[82385, 95867, … 18505]"
207,"[53916, 32744, … 96018]"


In [11]:
def combine_recommendations(history, fallback, k=12):
    recommendations = []
    seen = set()

    for item in (history or []) + fallback:
        if item not in seen:
            recommendations.append(item)
            seen.add(item)

        if len(recommendations) == k:
            break

    return recommendations

In [12]:
evaluation = validation_ground_truth.join(user_history, on="customer_idx", how="left")

predictions = [
    combine_recommendations(history, decay_top100, K)
    for history in evaluation["history"].to_list()
]

actuals_personal = evaluation["actual"].to_list()

In [13]:
def evaluate_personalized(actuals, predictions, catalog_size, k=12):
    return {
        "MAP@12": sum(average_precision_at_k(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals),
        "Recall@12": sum(recall_at_k(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals),
        "NDCG@12": sum(ndcg_at_k(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals),
        "Coverage": len({item for prediction in predictions for item in prediction[:k]}) / catalog_size
    }

In [14]:
history_decay_metrics = evaluate_personalized(
    actuals_personal,
    predictions,
    catalog_size
)

history_decay_metrics

{'MAP@12': 0.02526378315327509,
 'Recall@12': 0.050229068248646015,
 'NDCG@12': 0.03648223605264525,
 'Coverage': 0.21896496181614902}

In [15]:
best_decay = decay_results.row(0, named=True)

comparison = pl.DataFrame([
    {
        "model": "Recent popularity 14d",
        "MAP@12": 0.006995,
        "Recall@12": 0.022269,
        "NDCG@12": 0.013167,
        "Coverage": 0.000114
    },
    {
        "model": f"Decay popularity h={BEST_HALF_LIFE}d",
        "MAP@12": best_decay["MAP@12"],
        "Recall@12": best_decay["Recall@12"],
        "NDCG@12": best_decay["NDCG@12"],
        "Coverage": best_decay["Coverage"]
    },
    {
        "model": "Personal history 56d + recent popularity 14d",
        "MAP@12": 0.025170,
        "Recall@12": 0.049558,
        "NDCG@12": 0.036092,
        "Coverage": 0.218965
    },
    {
        "model": f"Personal history 56d + decay popularity h={BEST_HALF_LIFE}d",
        **history_decay_metrics
    }
]).sort("MAP@12", descending=True)

comparison

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Personal history 56d + decay p…",0.025264,0.050229,0.036482,0.218965
"""Personal history 56d + recent …",0.02517,0.049558,0.036092,0.218965
"""Decay popularity h=3d""",0.007,0.022814,0.013504,0.000114
"""Recent popularity 14d""",0.006995,0.022269,0.013167,0.000114
